In [ ]:
import pandas as pd

df_dggb_1 = pd.read_csv('./data/건봉국밥_리뷰_250527.csv')
df_dggb_2 = pd.read_csv('./data/대건명가_리뷰_250527.csv')
df_dggb_3 = pd.read_csv('./data/도야진국_리뷰_250527.csv')

df_gcgg_1 = pd.read_csv('./data/명장김치찌개_리뷰_250602.csv')
df_gcgg_2 = pd.read_csv('./data/신사강김치찌개_리뷰_250602.csv')
df_gcgg_3 = pd.read_csv('./data/금돼지식당_리뷰_250602.csv')

In [ ]:
# 각 데이터프레임에서 세부옵션 value counts 가장 높은 걸로만 슬라이싱하는 함수
def slice_top_option(df, option_column):
    try:
        top_value = df['세부옵션'].value_counts().idxmax()
    except:
        return df
    return df[df['세부옵션'] == top_value]

# 모든 데이터프레임에 함수 적용
df_dggb_1 = slice_top_option(df_dggb_1, '세부옵션')
df_dggb_2 = slice_top_option(df_dggb_2, '세부옵션')
df_dggb_3 = slice_top_option(df_dggb_3, '세부옵션')
df_gcgg_1 = slice_top_option(df_gcgg_1, '세부옵션')
df_gcgg_2 = slice_top_option(df_gcgg_2, '세부옵션')
df_gcgg_3 = slice_top_option(df_gcgg_3, '세부옵션')

In [ ]:
import re

def clean_text(text):
    # 특수문자 제거, 영어/숫자 선택적 포함 가능
    text = re.sub(r'[^가-힣\s]', ' ', text)  # 한글과 공백만 남기기
    text = re.sub(r'\s+', ' ', text).strip()  # 공백 정리
    return text

from konlpy.tag import Okt

okt = Okt()

def tokenize_and_lemmatize(text):
    # 명사, 형용사, 동사 중심으로 추출하고 원형 복원
    morphs = okt.pos(text, stem=True)  # stem=True -> 레마타이징 수행
    result = [word for word, tag in morphs if tag in ['Noun', 'Verb', 'Adjective']]
    return result

In [ ]:
# 전처리 적용 -> 시간 꽤 걸림
df_dggb_1['tokens'] = df_dggb_1['리뷰'].apply(clean_text).apply(tokenize_and_lemmatize)
df_dggb_2['tokens'] = df_dggb_2['리뷰'].apply(clean_text).apply(tokenize_and_lemmatize)
df_dggb_3['tokens'] = df_dggb_3['리뷰'].apply(clean_text).apply(tokenize_and_lemmatize)
df_gcgg_1['tokens'] = df_gcgg_1['리뷰'].apply(clean_text).apply(tokenize_and_lemmatize)
df_gcgg_2['tokens'] = df_gcgg_2['리뷰'].apply(clean_text).apply(tokenize_and_lemmatize)
df_gcgg_3['tokens'] = df_gcgg_3['리뷰'].apply(clean_text).apply(tokenize_and_lemmatize)

In [ ]:
def dedup_preserve_order(tokens):
    seen = set()
    result = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            result.append(token)
    return result

# 새로운 열 추가
df_dggb_1['tokens_dedup'] = df_dggb_1['tokens'].apply(dedup_preserve_order)
df_dggb_2['tokens_dedup'] = df_dggb_2['tokens'].apply(dedup_preserve_order)
df_dggb_3['tokens_dedup'] = df_dggb_3['tokens'].apply(dedup_preserve_order)
df_gcgg_1['tokens_dedup'] = df_gcgg_1['tokens'].apply(dedup_preserve_order)
df_gcgg_2['tokens_dedup'] = df_gcgg_2['tokens'].apply(dedup_preserve_order)
df_gcgg_3['tokens_dedup'] = df_gcgg_3['tokens'].apply(dedup_preserve_order)

# 맛있다 언급 수 분석

In [ ]:
from collections import Counter
from itertools import chain
import numpy as np

total_info_df = pd.DataFrame(columns=['가게명', '리뷰수', '세부옵션', '평균평점', '맛있다_수'])

for temp in [df_dggb_1, df_dggb_2, df_dggb_3, df_gcgg_1, df_gcgg_2, df_gcgg_3]:
    

    token_freq = Counter(chain.from_iterable(temp['tokens_dedup']))
    df_freq = pd.DataFrame(token_freq.items(), columns=['토큰', '빈도수'])
    print(df_freq.sort_values(by='빈도수', ascending=False).head(10))
    print('--' * 30)

    review_count = len(temp)
    try:
        option = temp['세부옵션'].value_counts().idxmax()
    except:
        option = np.nan

    mean_score = temp['별점'].mean()
    delicious_count = df_freq[df_freq['토큰'] == '맛있다'].values[0][1]
    
    total_info_df.loc[len(total_info_df)] = [np.nan, review_count, option, mean_score, delicious_count]

total_info_df['가게명'] = ['건봉국밥', '대건명가', '도야진국', '명장김치찌개', '신사강김치찌개', '금돼지식당']
total_info_df['맛있다_비율'] = total_info_df['맛있다_수'] / total_info_df['리뷰수']
total_info_df

In [ ]:
# 평점, 맛있다 비율 scartter plot
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Pretendard'
plt.figure(figsize=(10, 6))
plt.scatter(total_info_df['평균평점'], total_info_df['맛있다_비율'], s=100, alpha=0.7)
plt.xlabel('평균 평점')
plt.ylabel('맛있다 비율')
# 각 점에 이름 붙이기
for i, row in total_info_df.iterrows():
    plt.annotate(row['가게명'],
                 (row['평균평점'], row['맛있다_비율']),
                 textcoords="offset points",  # 좌표 기준이 점에서의 오프셋으로 변경됨
                 xytext=(0, 7),               # 위로 5포인트 이동
                 ha='left',
                 fontsize=10)
plt.title('평균 평점 vs 맛있다 비율')

# 특정 맛 표현 분석

In [ ]:
from collections import Counter
from itertools import chain
pd.set_option('display.max_rows', 10)  # 모든 행 보기
# pd.set_option('display.max_rows', 10)
# 모든 토큰 리스트 수집



all_tokens = list(chain.from_iterable(
    df['tokens'] for df in [df_dggb_1, df_dggb_2, df_dggb_3]  # 또는 'tokens_dedup'
))

# 빈도수 계산
token_freq = Counter(chain.from_iterable(all_tokens))

# DataFrame으로 보기 좋게 변환
freq_df = pd.DataFrame(token_freq.items(), columns=['토큰', '빈도수']).sort_values(by='빈도수', ascending=False)
freq_df

In [ ]:
from collections import Counter
from itertools import chain
pd.set_option('display.max_rows', 10)  # 모든 행 보기
# pd.set_option('display.max_rows', 10)
# 모든 토큰 리스트 수집



all_tokens = list(chain.from_iterable(
    df['tokens'] for df in [df_gcgg_1, df_gcgg_2, df_gcgg_3]  # 또는 'tokens_dedup'
))

# 빈도수 계산
token_freq = Counter(chain.from_iterable(all_tokens))

# DataFrame으로 보기 좋게 변환
freq_df = pd.DataFrame(token_freq.items(), columns=['토큰', '빈도수']).sort_values(by='빈도수', ascending=False)
freq_df

In [ ]:
from collections import Counter
from itertools import chain
import numpy as np
import pandas as pd

# 맛 관련 키워드 리스트
taste_keywords = [
    "깔끔하다", "진하다", "넉넉하다", "진국", "든든하다", "담백하다",
    "부드럽다", "구수하다", "뜨끈하다", "고소하다", "비리다", "짜다",
    "느끼하다", "시원하다", "신선하다", "깊다", "맵다", "매콤"
]

# 결과 저장용 컬럼 정의
base_columns = ['가게명', '리뷰수', '세부옵션', '평균평점', '맛있다_수', '맛있다_비율']
keyword_columns = [f"{kw}_비율" for kw in taste_keywords]
total_info_df = pd.DataFrame(columns=base_columns + keyword_columns)

# 각 가게별 처리
for temp in [df_dggb_1, df_dggb_2, df_dggb_3, df_gcgg_1, df_gcgg_2, df_gcgg_3]:

    # 전체 토큰 빈도
    token_freq = Counter(chain.from_iterable(temp['tokens_dedup']))
    df_freq = pd.DataFrame(token_freq.items(), columns=['토큰', '빈도수'])
    
    review_count = len(temp)
    try:
        option = temp['세부옵션'].value_counts().idxmax()
    except:
        option = np.nan

    mean_score = temp['별점'].mean()
    delicious_count = df_freq[df_freq['토큰'] == '맛있다'].values[0][1] if '맛있다' in df_freq['토큰'].values else 0
    delicious_ratio = delicious_count / review_count if review_count > 0 else 0

    # 각 키워드별 비율 계산
    keyword_ratios = []
    for kw in taste_keywords:
        kw_count = temp['tokens_dedup'].apply(lambda tokens: kw in tokens).sum()
        kw_ratio = kw_count / review_count if review_count > 0 else 0
        keyword_ratios.append(kw_ratio)

    # 한 줄로 정리해 추가
    total_info_df.loc[len(total_info_df)] = [np.nan, review_count, option, mean_score,
                                              delicious_count, delicious_ratio] + keyword_ratios

# 가게명 기입
total_info_df['가게명'] = ['건봉국밥', '대건명가', '도야진국', '명장김치찌개', '신사강김치찌개', '금돼지식당']

# 결과 확인
total_info_df

In [ ]:
total_info_df.columns

In [ ]:
pd.set_option('display.max_rows', None)
ratio_taste_df = total_info_df[['가게명', '평균평점', '깔끔하다_비율', '진하다_비율',
       '넉넉하다_비율', '진국_비율', '든든하다_비율', '담백하다_비율', '부드럽다_비율', '구수하다_비율',
       '뜨끈하다_비율', '고소하다_비율', '비리다_비율', '짜다_비율', '느끼하다_비율', '시원하다_비율',
       '신선하다_비율', '깊다_비율', '맵다_비율', '매콤_비율']].T
ratio_taste_df.columns = total_info_df['가게명']
ratio_taste_df = ratio_taste_df.reset_index().rename(columns={'index': '맛 관련 키워드'})
ratio_taste_df.drop(index=0, inplace=True)
ratio_taste_df = ratio_taste_df.set_index('맛 관련 키워드')
ratio_taste_df